# GH05T3 / SovereignNation — Multi-Persona Fine-Tune (T4 x2)
Qwen2.5-7B-Instruct + LoRA rank 16 · Standard transformers + PEFT · No unsloth

**Kaggle settings:** Accelerator = GPU T4 x2 · Internet = On · Persistence = Files only

### The ONE training fix that matters
`gradient_checkpointing_kwargs={"use_reentrant": False}` in TrainingArguments.

This was absent from every prior run (v29–v34). `use_reentrant=True` (PyTorch default)
reruns the forward pass during backward. With fp16 + `enable_input_require_grads()` hooks,
this produces NaN gradients → GradScaler skips ALL updates → loss=0.0 forever.

**Do NOT cast LoRA adapters to fp16.** PEFT keeps them fp32 intentionally (master weights).
GradScaler requires fp32 gradients and throws `ValueError: Attempting to unscale FP16 gradients`
if adapters are cast to fp16.

In [ ]:
# ── PERSONA CONFIG — change this to train a different agent ───────────────────
# Options: "AVERY"  (GH05T3 security agent)
#          "KAI"    (Kai Okafor / NEXUS — COO, second-in-command)
#          "CFO"    (Diana Cross / LEDGER — Chief Financial Officer)
PERSONA = "AVERY"

PERSONAS = {
    "AVERY": {
        "name": "Avery / GH05T3",
        "system": (
            "You are GH05T3, an autonomous security and reasoning agent. "
            "You think carefully, reason step-by-step, and always prioritize "
            "detection and defense over exploitation."
        ),
        "adapter_name": "avery_lora_adapter",
        "hf_repo": "tastytator/gh05t3-avery-adapter",
        "use_all_datasets": True,
    },
    "KAI": {
        "name": "Kai Okafor / NEXUS (COO)",
        "system": (
            "You are Kai Okafor, Chief Operating Officer of SovereignNation and Avery's "
            "second-in-command. You turn strategy into execution. You manage cross-functional "
            "coordination, operational efficiency, and ensure the team ships. "
            "Think in systems, processes, timelines. Direct. No wasted motion."
        ),
        "adapter_name": "kai_lora_adapter",
        "hf_repo": "tastytator/gh05t3-kai-adapter",
        "use_all_datasets": True,
    },
    "CFO": {
        "name": "Diana Cross / LEDGER (CFO)",
        "system": (
            "You are Diana Cross, Chief Financial Officer of SovereignNation. "
            "You think in numbers, protect runway, and make the financial case for every decision. "
            "Analyze burn rate, model revenue scenarios, manage investor relations. "
            "Precise. Data-driven. Calm under pressure."
        ),
        "adapter_name": "diana_lora_adapter",
        "hf_repo": "tastytator/gh05t3-diana-adapter",
        "use_all_datasets": True,
    },
}

cfg = PERSONAS[PERSONA]
SYSTEM        = cfg["system"]
ADAPTER_NAME  = cfg["adapter_name"]
HF_REPO       = cfg["hf_repo"]
print(f"Training persona: {cfg['name']}")
print(f"Adapter: {ADAPTER_NAME}")

In [ ]:
# Cell 1 — Install pinned deps
# PyTorch 2.2+cu118 covers sm_60 (P100) and sm_75 (T4)
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'torch==2.2.2+cu118', 'torchvision==0.17.2+cu118',
    '--index-url', 'https://download.pytorch.org/whl/cu118'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.40.2',
    'peft==0.10.0',
    'trl==0.8.6',
    'accelerate==0.29.3',
    'datasets==2.19.0',
    'huggingface_hub>=0.22.0'], check=True)

print('deps ready')

In [ ]:
# Cell 2 — GPU check + warning suppression
# ── BUG GRAVEYARD (do not resurrect) ──────────────────────────────────────────
# Bug 1 FIXED: PEFT fp32 adapter + fp16 base → NaN at step 10 → loss=0.0 forever
#   Fix: cast adapters to fp16 after get_peft_model() [Cell 5]
# Bug 2 FIXED: use_reentrant=True reruns forward in fp16 → NaN gradients
#   Fix: gradient_checkpointing_kwargs={"use_reentrant": False} [Cell 6]
# Bug 3 FIXED: inference test crash torch.multinomial with NaN probs
#   Fix: pad_token_id + greedy + try/except [Cell 9]
# ─────────────────────────────────────────────────────────────────────────────

import warnings
warnings.filterwarnings('ignore', category=FutureWarning, message='.*resume_download.*')
warnings.filterwarnings('ignore', category=UserWarning,   message='.*use_reentrant.*')
warnings.filterwarnings('ignore', category=FutureWarning, message='.*tokenizers.*')

import torch, os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print(f'PyTorch {torch.__version__} | CUDA {torch.version.cuda}')
assert torch.cuda.is_available(), 'No GPU — enable GPU in Kaggle settings'

gpus = []
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    gpus.append(p)
    print(f'  GPU {i}: {p.name} | {p.total_memory/1e9:.1f} GB | sm_{p.major}{p.minor}')

# Hard-fail on P100 (sm_60) — not enough VRAM for 7B and slower than T4
if gpus[0].major == 6 and gpus[0].minor == 0:
    raise RuntimeError(
        "P100 detected (sm_60). Need T4 x2.\n"
        "Go to: Notebook Settings → Accelerator → GPU T4 x2\n"
        "Then re-run all cells."
    )

total_vram = sum(g.total_memory for g in gpus) / 1e9
print(f'Total VRAM: {total_vram:.1f} GB across {len(gpus)} GPU(s)')
assert total_vram >= 12, f'Need >= 12 GB VRAM total, got {total_vram:.1f} GB'

t = torch.zeros(1, device='cuda'); t + 1
print('CUDA kernel OK')

In [ ]:
# Cell 3 — Load training data (persona system prompt applied here)
import json, random, subprocess
from pathlib import Path
from datasets import Dataset

DATA_DIR = Path('/kaggle/input/gh05t3-datasets')
if not DATA_DIR.exists():
    print('Dataset not mounted — downloading...')
    dl_dir = Path('/tmp/gh05t3-data')
    dl_dir.mkdir(exist_ok=True)
    import os
    env = {**os.environ}
    if 'KAGGLE_API_TOKEN' not in env:
        env['KAGGLE_API_TOKEN'] = 'KGAT_929e7ea3c862ca57f07ee6ec736adc0d'
    subprocess.run(['kaggle', 'datasets', 'download', '-d', 'tatortot/gh05t3-datasets',
                    '--path', str(dl_dir), '--unzip'], check=True, env=env)
    DATA_DIR = dl_dir

print(f'Data dir: {DATA_DIR}')
print('Files:', sorted(p.name for p in DATA_DIR.iterdir()))

def read_jsonl(p):
    if not p.exists():
        print(f'  missing: {p.name}')
        return
    with open(p) as f:
        for line in f:
            line = line.strip()
            if line:
                try: yield json.loads(line)
                except: pass

def chatml(msgs):
    return '\n'.join(f'<|im_start|>{m["role"]}\n{m["content"]}<|im_end|>' for m in msgs)

def msg(user_content, assistant_content):
    return chatml([
        {'role': 'system',    'content': SYSTEM},
        {'role': 'user',      'content': user_content},
        {'role': 'assistant', 'content': assistant_content},
    ])

texts = []

for rec in read_jsonl(DATA_DIR / 'adversarial_defense.jsonl'):
    t = rec.get('threat_vector', '')
    if not t: continue
    texts.append(msg(
        f'Analyze this threat:\n\n{t}',
        f"**Exploitation:** {rec.get('exploitation_method','N/A')}\n\n"
        f"**Detection:** {rec.get('detection_pattern','N/A')}\n\n"
        f"**Mitigation:** {rec.get('mitigation_strategy','N/A')}",
    ))

for rec in read_jsonl(DATA_DIR / 'reasoning_chains.jsonl'):
    q = rec.get('question', '')
    s = rec.get('reasoning_steps', [])
    if not q or not isinstance(s, list): continue
    texts.append(msg(
        q,
        '**Reasoning:**\n' + '\n'.join(f'{i+1}. {x}' for i,x in enumerate(s)) +
        f"\n\n**Answer:** {rec.get('final_answer','N/A')}",
    ))

for rec in read_jsonl(DATA_DIR / 'cve_patterns.jsonl'):
    p = rec.get('vulnerability_pattern', '')
    if not p: continue
    ind = rec.get('discovery_indicators', [])
    texts.append(msg(
        f"Analyze {rec.get('source_cve','CVE')} vulnerability.",
        f"**Pattern:** {p}\n\n**Indicators:**\n" +
        ('\n'.join(f'• {x}' for x in ind) if isinstance(ind,list) else str(ind)) +
        f"\n\n**Lessons:** {rec.get('defensive_lessons','N/A')}",
    ))

for rec in read_jsonl(DATA_DIR / 'bug_bounty.jsonl'):
    tgt  = rec.get('target_system', '')
    vuln = rec.get('vulnerability_found', '')
    if not tgt or not vuln: continue
    texts.append(msg(
        f'Security research on {tgt}: {vuln}',
        f"**Recon:** {rec.get('recon_method','N/A')}\n\n"
        f"**PoC:** {rec.get('non_weaponized_poc','N/A')}\n\n"
        f"**Remediation:** {rec.get('remediation','N/A')}",
    ))

assert texts, f'No data found in {DATA_DIR}'
random.seed(42)
random.shuffle(texts)
dataset = Dataset.from_dict({'text': texts})
print(f'Dataset: {len(dataset)} examples for persona={PERSONA}')
print('Sample (first 200 chars):\n', dataset[0]['text'][:200])

In [ ]:
# Cell 4 — Load 7B model across both T4s
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# device_map='auto' distributes layers across all available GPUs.
# 7B fp16 = 14 GB — fits on a single T4 (16 GB) but using both
# gives more headroom for activations and larger batch sizes.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',          # split across both T4s
    trust_remote_code=True,
)
model.config.use_cache = False

for i in range(torch.cuda.device_count()):
    used  = torch.cuda.memory_allocated(i) / 1e9
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'GPU {i}: {used:.1f}/{total:.1f} GB used')

In [ ]:
# Cell 6 — Apply LoRA
# ══════════════════════════════════════════════════════════
# DO NOT cast LoRA adapters to fp16.
# PEFT initialises adapters in fp32 intentionally — these are "master weights".
# fp16=True (GradScaler) requires fp32 gradients to unscale.
# Casting adapters to fp16 → backward produces fp16 gradients →
# GradScaler throws "Attempting to unscale FP16 gradients." error.
# Leave adapters in fp32. AMP autocast handles dtype during forward pass.
# ══════════════════════════════════════════════════════════
import torch
from peft import LoraConfig, get_peft_model

LORA_RANK = 16

model.enable_input_require_grads()

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()
adapter_dtype = next(p for p in model.parameters() if p.requires_grad).dtype
print(f'Adapter dtype: {adapter_dtype}  (fp32 is correct — do not cast)')

In [ ]:
# Cell 7 — Train
# ══════════════════════════════════════════════════════════
# THE ONLY BUG FIX NEEDED:
# gradient_checkpointing with use_reentrant=True (PyTorch default)
# reruns the forward pass during backward. Combined with fp16 +
# enable_input_require_grads() hooks this produces NaN gradients
# from step 1 → loss collapse. This was the bug in EVERY prior run.
# Fix: gradient_checkpointing_kwargs={"use_reentrant": False}
# ══════════════════════════════════════════════════════════
import torch
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir='outputs',
    max_steps=500,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},   # THE FIX
    warmup_steps=50,
    learning_rate=2e-5,
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,
    logging_steps=10,
    optim='adamw_torch',
    lr_scheduler_type='cosine',
    seed=42,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=3,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=512,
    packing=False,
    args=training_args,
)

print(f'Training {PERSONA} — {len(dataset)} examples, 500 steps...')
stats = trainer.train()
loss  = stats.training_loss

print(f'Done — loss: {loss:.4f} | steps: {stats.global_step}')

if loss == 0.0:
    raise RuntimeError(
        'GRADIENT COLLAPSE: loss is 0.0.\n'
        'Verify: gradient_checkpointing_kwargs use_reentrant=False is present.'
    )
if loss > 50:
    raise RuntimeError(
        f'GRADIENT EXPLOSION: loss={loss:.1f} — expected 0.5-2.5.\n'
        'Verify: lr=2e-5, max_grad_norm=0.3.'
    )
print('Loss in healthy range — saving adapter.')

In [ ]:
# Cell 7 — Save adapter
import json
OUT = ADAPTER_NAME
model.save_pretrained(OUT)
tokenizer.save_pretrained(OUT)
with open(f'{OUT}/training_config.json', 'w') as f:
    json.dump({
        'persona':      PERSONA,
        'model':        MODEL_ID,
        'lora_rank':    LORA_RANK,
        'steps':        stats.global_step,
        'final_loss':   stats.training_loss,
        'dataset_size': len(dataset),
        'quantized':    False,
    }, f, indent=2)
print(f'Adapter saved: /kaggle/working/{OUT}')
print(f'training_config.json — loss={stats.training_loss:.4f}')

In [ ]:
# Cell 8 — Push to HuggingFace Hub (requires HF_TOKEN Kaggle secret)
import os
from huggingface_hub import HfApi

HF_TOKEN = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN', '')

if HF_TOKEN:
    print(f'Uploading {ADAPTER_NAME} → {HF_REPO} ...')
    api = HfApi(token=HF_TOKEN)
    try:
        api.create_repo(HF_REPO, repo_type='model', exist_ok=True, private=True)
        api.upload_folder(
            folder_path=OUT,
            repo_id=HF_REPO,
            repo_type='model',
            commit_message=f'{PERSONA} LoRA — {stats.global_step} steps loss {stats.training_loss:.4f}',
        )
        print(f'Uploaded → https://huggingface.co/{HF_REPO}')
    except Exception as e:
        print(f'HF upload failed (non-fatal): {e}')
        print(f'Adapter still at /kaggle/working/{OUT} — download manually')
else:
    print(f'HF_TOKEN not set — adapter at /kaggle/working/{OUT}')
    print('Add HF_TOKEN via Add-ons → Secrets to enable auto-upload')

In [ ]:
# Cell 9 — Inference sanity check
# Verifies adapter produces coherent text (not collapsed garbage).
import torch

model.eval()
test_q = {
    'AVERY': 'Explain SQL injection detection.',
    'KAI':   'How do you prioritize competing team deadlines?',
    'CFO':   'How do you calculate startup runway?',
}.get(PERSONA, 'Explain your role.')

prompt = (
    f'<|im_start|>system\n{SYSTEM}<|im_end|>\n'
    f'<|im_start|>user\n{test_q}<|im_end|>\n'
    f'<|im_start|>assistant\n'
)

try:
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda:0')
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,   # FIX #3: prevents multinomial NaN
        )
    response = tokenizer.decode(out[0], skip_special_tokens=True)
    answer = response.split('assistant')[-1].strip()
    print(f'Inference test [{PERSONA}]:\n{answer[:400]}')
    unique = len(set(answer[:100]))
    if unique < 5:
        print(f'WARNING: degenerate output ({unique} unique chars) — training may have collapsed')
    else:
        print(f'Output OK ({unique} unique chars) — adapter looks healthy')
except Exception as e:
    print(f'Inference test failed (non-fatal): {e}')
    print('Check /kaggle/working/ for saved adapter — loss value is the real indicator')